[Abrir esta atividade no Colab](https://colab.research.google.com/github/MRCahu/RNA-e-Deep-Learning---CESAR-School-2026.01-A/blob/main/Atividade%202/atividade_2.ipynb)

# Atividade 2 — Regressão linear multivariada em PyTorch

**Aluno:** Mauro Roberto Barbosa Cahu · **E-mail:** mrbc@cesar.school  
**Disciplina:** RNA e Deep Learning — CESAR School 2026.01-A  
**Professor:** Vitor Casadei  
**Problema:** prever o valor mediano das moradias de grupos de quarteirões da Califórnia com oito entradas.

## Objetivos e relação com o notebook 2.6

Esta atividade adapta o pipeline do [notebook-base oficial 2.6_regressao_linear_salarios.ipynb](https://github.com/vcasadei/Redes-Neurais-CESAR-School/blob/2026.01-A/2%20-%20Regress%C3%A3o%20Linear/2.6_regressao_linear_salarios.ipynb).

Mantêm-se a sequência didática, as funções `train_epoch` e `validate_epoch`, `nn.Linear`, `nn.MSELoss` e `optim.SGD`. A tarefa exige adaptar textos, dados e análises: oito entradas, três conjuntos, RMSE em dólares, baseline e interpretação dos oito pesos. O original permanece inalterado; as saídas deste notebook são produzidas novamente para o California Housing.

Roteiro: dados → separação → padronização → modelo → treinamento → avaliação → interpretação → conclusão.

Cada linha representa um **grupo de quarteirões do Censo de 1990**, não uma casa individual. O alvo é o **valor mediano das moradias desse grupo**, em centenas de milhares de dólares: `y = 2` significa US$ 200.000. Não se trata de uma cotação atual.

O modelo combina oito informações:

$$\hat{y}_{pad}=b+\sum_{j=1}^{8} w_j x_{j,pad}.$$

Exemplo prático: uma planilha contém oito características do bairro e uma coluna com o preço observado. O treino ajusta oito pesos e um intercepto para aproximar essa última coluna. Não há camada oculta nem função de ativação: é regressão linear.

## 1. Bibliotecas e reprodutibilidade

No Google Colab, as bibliotecas abaixo normalmente já estão instaladas. Em ambiente próprio, use o `requirements.txt` desta pasta. A execução usa CPU, sementes fixas e uma thread; versões diferentes podem produzir pequenas diferenças numéricas.

In [ ]:
import copy
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import torch
from torch import nn, optim
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from IPython.display import display, Markdown

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})
VERSOES = {'numpy': np.__version__, 'pandas': pd.__version__,
           'matplotlib': __import__('matplotlib').__version__,
           'scikit-learn': sklearn.__version__, 'torch': torch.__version__}
print('Python:', sys.version.split()[0], '| dispositivo: CPU')
display(pd.Series(VERSOES, name='Versão efetivamente executada').to_frame())

## 2. Dados — obtenção sem login

Usamos `fetch_california_housing(as_frame=True)`, conforme o enunciado. O primeiro uso precisa de internet; o scikit-learn mantém um cache nas próximas execuções. Não substituímos a base por dados sintéticos.

Fonte e dicionário: [scikit-learn — California Housing](https://scikit-learn.org/stable/datasets/real_world.html#california-housing-dataset). O retorno `dados.DESCR` registra a descrição da fonte.

In [ ]:
dados = fetch_california_housing(as_frame=True)
X, y = dados.data.copy(), dados.target.copy()
assert X.shape == (20640, 8)
assert y.shape == (20640,)
print(dados.DESCR)

### Dicionário, inspeção e estatísticas descritivas

As entradas são agregadas por grupo de quarteirões. `AveRooms` inclui os cômodos; `AveBedrms` refere-se aos dormitórios. Renda mediana e preço mediano são conceitos diferentes. As análises descritivas completas atendem ao enunciado; o ajuste e a escolha da época usam apenas treino e validação.

In [ ]:
dicionario = pd.DataFrame([
    ('MedInc', 'Renda mediana dos domicílios', 'unidades de US$ 10.000'),
    ('HouseAge', 'Idade mediana das moradias', 'anos'),
    ('AveRooms', 'Média de cômodos por domicílio', 'cômodos/domicílio'),
    ('AveBedrms', 'Média de dormitórios por domicílio', 'dormitórios/domicílio'),
    ('Population', 'População do grupo de quarteirões', 'pessoas'),
    ('AveOccup', 'Média de moradores por domicílio', 'pessoas/domicílio'),
    ('Latitude', 'Coordenada geográfica norte–sul', 'graus'),
    ('Longitude', 'Coordenada geográfica leste–oeste', 'graus'),
    ('MedHouseVal (alvo)', 'Valor mediano das moradias', 'US$ 100.000'),
], columns=['Variável', 'Significado', 'Unidade'])
display(dicionario)
df = X.assign(MedHouseVal=y)
display(df.head())
display(df.describe().T)
print('Valores ausentes:', int(df.isna().sum().sum()))
print('Linhas integralmente duplicadas:', int(df.duplicated().sum()))
print('Valores não finitos:', int((~np.isfinite(df.to_numpy())).sum()))
print('Alvo máximo em dólares:', f'US$ {y.max() * 100_000:,.0f}')
print('Registros no alvo máximo:', int((y == y.max()).sum()))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, coluna, titulo in zip(axes, ['MedInc', 'AveRooms'],
                            ['Renda mediana', 'Média de cômodos por domicílio']):
    ax.scatter(X[coluna], y * 100_000, s=7, alpha=0.16, color='#2266aa', rasterized=True)
    ax.set(xlabel=coluna, ylabel='Valor mediano das moradias (US$)', title=titulo)
    ax.ticklabel_format(style='plain', axis='y')
    ax.grid(alpha=0.2)
fig.suptitle('California Housing — cada ponto é um grupo de quarteirões')
fig.tight_layout()
plt.show()
display(Markdown('**Leitura:** a renda apresenta tendência positiva com o preço, mas há '
                 'dispersão: renda sozinha não determina o valor. Em `AveRooms`, valores '
                 'extremos comprimem a nuvem principal; o eixo completo foi mantido. '
                 'O alinhamento horizontal no topo indica o teto do alvo, perto de '
                 'US$ 500.001. Não removemos esses registros nem os extremos.'))

## 3. Separação em treino, validação e teste — 70/15/15

**Treino:** ajusta os pesos e os scalers. **Validação:** escolhe a melhor época. **Teste:** mede o resultado final uma única vez, depois da seleção. Dois sorteios com `random_state=42` mantêm grupos disjuntos: 70% para treino; os 30% restantes são divididos igualmente. Os índices originais permitem verificar que nenhuma linha aparece em dois conjuntos.

A divisão aleatória atende à atividade. Como há proximidade geográfica entre observações, ela não prova desempenho em regiões completamente novas; uma melhoria seria validação por blocos geográficos.

In [ ]:
indices = np.arange(len(X))
idx_train, idx_temp = train_test_split(indices, test_size=0.30, random_state=SEED)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.50, random_state=SEED)
x_raw = X.to_numpy()
y_raw = y.to_numpy().reshape(-1, 1)
x_train_raw, y_train_raw = x_raw[idx_train], y_raw[idx_train]
x_val_raw, y_val_raw = x_raw[idx_val], y_raw[idx_val]
x_test_raw, y_test_raw = x_raw[idx_test], y_raw[idx_test]
assert not (set(idx_train) & set(idx_val) or set(idx_train) & set(idx_test) or set(idx_val) & set(idx_test))
assert len(set(indices)) == len(idx_train) + len(idx_val) + len(idx_test)
display(pd.DataFrame({'Conjunto': ['Treino', 'Validação', 'Teste'],
                      'Linhas': [len(idx_train), len(idx_val), len(idx_test)],
                      'Percentual': [70, 15, 15]}))

## 4. Padronização das entradas e da saída

$$z=(x-\mu_{treino})/\sigma_{treino}.$$

Cada coluna ganha sua própria régua: zero é a média do treino e uma unidade corresponde a um desvio padrão do treino. `fit` aprende a régua; `transform` a aplica; `inverse_transform` desfaz a operação.

Usamos **dois scalers**, um para X e outro para y, ambos ajustados **somente no treino**. Validação e teste recebem exclusivamente `transform`. O alvo permanece em unidades de US$ 100.000 antes de ser padronizado. Na avaliação, primeiro desfazemos a padronização e depois multiplicamos por 100.000.

In [ ]:
x_scaler = StandardScaler().fit(x_train_raw)
y_scaler = StandardScaler().fit(y_train_raw)

def tensor_padronizado(scaler, valores):
    return torch.from_numpy(scaler.transform(valores)).float()

x_train = tensor_padronizado(x_scaler, x_train_raw)
y_train = tensor_padronizado(y_scaler, y_train_raw)
x_val = tensor_padronizado(x_scaler, x_val_raw)
y_val = tensor_padronizado(y_scaler, y_val_raw)
x_test = tensor_padronizado(x_scaler, x_test_raw)
y_test = tensor_padronizado(y_scaler, y_test_raw)
assert x_scaler.n_samples_seen_ == len(idx_train)
assert y_scaler.n_samples_seen_ == len(idx_train)
np.testing.assert_allclose(x_scaler.mean_, x_train_raw.mean(axis=0))
np.testing.assert_allclose(y_scaler.mean_, y_train_raw.mean(axis=0))
print('X treino / validação / teste:', x_train.shape, x_val.shape, x_test.shape)
print('y treino / validação / teste:', y_train.shape, y_val.shape, y_test.shape)
display(pd.DataFrame({'Variável': X.columns, 'Média do treino': x_scaler.mean_,
                      'Desvio padrão do treino': x_scaler.scale_}))
print('Média do alvo no treino:', float(y_scaler.mean_[0]))
print('Desvio padrão do alvo no treino:', float(y_scaler.scale_[0]))

## 5. Modelo — `nn.Linear(8, 1)`

São oito entradas, uma saída e **nove parâmetros**: oito pesos mais um intercepto. Cada linha de X tem oito números; o modelo devolve um preço padronizado. O bias já está incluído na camada, sem coluna adicional de uns.

In [ ]:
torch.manual_seed(SEED)
model = nn.Linear(8, 1)
print(model)
print('Formato dos pesos:', tuple(model.weight.shape))
print('Formato do bias:', tuple(model.bias.shape))
print('Número de parâmetros:', sum(p.numel() for p in model.parameters()))

### Teste de formato antes do treinamento

Esta previsão inicial usa pesos aleatórios e ainda não deve ser interpretada como estimativa útil.

In [ ]:
with torch.no_grad():
    print('Cinco linhas com oito entradas geram:', model(torch.ones(5, 8)).shape)

In [ ]:
with torch.no_grad():
    print('Previsões iniciais, em escala padronizada:')
    print(model(x_train[:3]))

## 6. Treinamento — custo e otimizador

`nn.MSELoss` calcula a média dos erros ao quadrado. `optim.SGD` atualiza pesos e bias usando os gradientes. Como no 2.6, cada passo usa o treino completo (full batch): o otimizador continua sendo SGD.

Fixamos `lr=0.05` e 4.000 épocas como orçamento de treino, sem consultar o teste. A taxa é menor que a do exemplo univariado porque entradas correlacionadas alteram a curvatura do custo. A melhor época é a de menor custo de validação.

In [ ]:
lr = 0.05
n_epochs = 4000
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=lr)

### Funções de treino e validação

`zero_grad()` limpa o gradiente anterior; `backward()` calcula os novos gradientes; `step()` aplica a atualização. A validação usa `model.eval()` e `torch.no_grad()` e não muda parâmetros.

In [ ]:
def train_epoch(model, optimizer, criterion, x_train, y_train):
    model.train()
    optimizer.zero_grad()
    y_pred = model(x_train)
    loss = criterion(y_pred, y_train)
    loss.backward()
    optimizer.step()
    return loss.item()

In [ ]:
def validate_epoch(model, criterion, x_val, y_val):
    model.eval()
    with torch.no_grad():
        return criterion(model(x_val), y_val).item()

### Laço de treinamento e seleção pela validação

Os dois custos das curvas são medidos **depois do mesmo passo de atualização**, para permitir comparação. Uma cópia independente dos parâmetros é guardada sempre que a validação melhora. Ao final, restauramos esse estado antes de avaliar o teste.

In [ ]:
train_losses, val_losses = [], []
best_val = float('inf')
best_epoch = None
best_state = None
for epoch in range(1, n_epochs + 1):
    train_epoch(model, optimizer, criterion, x_train, y_train)
    train_loss = validate_epoch(model, criterion, x_train, y_train)
    val_loss = validate_epoch(model, criterion, x_val, y_val)
    if not np.isfinite([train_loss, val_loss]).all():
        raise RuntimeError('Treinamento divergiu: rever a taxa usando apenas treino e validação.')
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    if val_loss < best_val:
        best_val, best_epoch = val_loss, epoch
        best_state = copy.deepcopy(model.state_dict())
    if epoch == 1 or epoch % 500 == 0:
        print(f'Época {epoch:4d}/{n_epochs} | MSE treino: {train_loss:.6f} | validação: {val_loss:.6f}')
model.load_state_dict(best_state)
print(f'Estado restaurado: época {best_epoch}, MSE de validação = {best_val:.6f}')

## 7. Avaliação — curvas de custo

O eixo vertical é MSE na escala **padronizada**, não dólares. A linha vertical identifica a época selecionada. Uma validação menor que o treino pode acontecer pela composição das amostras; isso, isoladamente, não demonstra vazamento.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
epocas = np.arange(1, n_epochs + 1)
for ax in axes:
    ax.plot(epocas, train_losses, label='Treino', color='#2266aa')
    ax.plot(epocas, val_losses, label='Validação', color='#c46000')
    ax.axvline(best_epoch, color='#444444', linestyle='--', label=f'Melhor época: {best_epoch}')
    ax.set(xlabel='Época', ylabel='MSE padronizado')
    ax.grid(alpha=0.25)
    ax.legend()
axes[0].set_title('Treinamento completo')
axes[1].set(xlim=(1, 200), title='Detalhe das primeiras 200 épocas')
fig.tight_layout()
plt.show()

### Valores observados × previstos — validação

Com oito entradas, não há uma única reta em um gráfico de duas dimensões que represente todo o modelo. Por isso, comparamos preço observado com previsto. Na diagonal, a previsão é exata; quanto mais longe dela, maior o erro.

In [ ]:
def prever_dolares(model, entradas_padronizadas):
    model.eval()
    with torch.no_grad():
        pred_pad = model(entradas_padronizadas).cpu().numpy()
    return y_scaler.inverse_transform(pred_pad).ravel().astype(np.float64) * 100_000

pred_val_usd = prever_dolares(model, x_val)
y_val_usd = y_val_raw.ravel() * 100_000
rmse_val = float(np.sqrt(mean_squared_error(y_val_usd, pred_val_usd)))
fig, ax = plt.subplots(figsize=(6.5, 5))
ax.scatter(y_val_usd, pred_val_usd, alpha=0.25, s=10, color='#2266aa')
limites = [min(y_val_usd.min(), pred_val_usd.min()), max(y_val_usd.max(), pred_val_usd.max())]
ax.plot(limites, limites, 'k--', label='Previsão perfeita')
ax.set(xlabel='Valor observado (US$)', ylabel='Valor previsto (US$)', title='Validação — observado × previsto')
ax.ticklabel_format(style='plain')
ax.legend()
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()
print(f'RMSE da validação: US$ {rmse_val:,.2f}')

## 8. Os oito pesos aprendidos

Cada peso mede a mudança prevista no **alvo padronizado** ao aumentar a entrada em **um desvio padrão do treino**, mantendo as outras sete entradas fixas. Multiplicar o peso pelo desvio padrão de y e por 100.000 traduz esse efeito para dólares.

Os pesos podem ser comparados diretamente quanto à magnitude **porque as oito entradas foram padronizadas**. Sem isso, compararíamos o efeito de um grau geográfico com o de uma pessoa ou um ano. Padronizar y muda todos os pesos pela mesma escala; não muda sua ordenação por magnitude.

Trata-se de associação condicional, **não de causalidade** nem de importância universal. Variáveis correlacionadas podem compartilhar efeitos e alterar sinais; latitude e longitude representam um plano geográfico conjunto, não uma regra universal sobre morar ao norte ou ao leste.

In [ ]:
pesos = model.weight.detach().cpu().numpy().ravel().astype(np.float64)
bias = float(model.bias.detach().item())
coeficientes = pd.DataFrame({'Variável': X.columns, 'Peso padronizado': pesos,
    'Efeito em US$ por +1 desvio padrão de X': pesos * y_scaler.scale_[0] * 100_000})
coeficientes['Magnitude'] = np.abs(pesos)
display(coeficientes.sort_values('Magnitude', ascending=False).reset_index(drop=True))
print(f'Intercepto padronizado: {bias:.6f}')
print('MSE treino no estado selecionado:', validate_epoch(model, criterion, x_train, y_train))
print('MSE validação no estado selecionado:', validate_epoch(model, criterion, x_val, y_val))
ordenados = coeficientes.sort_values('Peso padronizado')
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(ordenados['Variável'], ordenados['Peso padronizado'],
        color=['#c46000' if v < 0 else '#2266aa' for v in ordenados['Peso padronizado']])
ax.axvline(0, color='black', linewidth=1)
ax.set(xlabel='Peso: efeito no alvo padronizado por +1 desvio padrão de X',
       title='Oito pesos — associação mantendo as demais entradas fixas')
ax.grid(axis='x', alpha=0.2)
fig.tight_layout()
plt.show()
positivos = coeficientes[coeficientes['Peso padronizado'] > 0].sort_values('Peso padronizado', ascending=False)
negativos = coeficientes[coeficientes['Peso padronizado'] < 0].sort_values('Peso padronizado')
print('Aumentam o preço previsto (maior efeito primeiro):', ', '.join(positivos['Variável']))
print('Diminuem o preço previsto (maior redução primeiro):', ', '.join(negativos['Variável']))
print('Maior aumento:', positivos.iloc[0]['Variável'])
print('Maior redução:', negativos.iloc[0]['Variável'])

## 9. Respostas à atividade e verificações

### RMSE no teste em dólares e baseline da média do treino

Só agora consultamos as previsões do teste. O baseline ignora as oito entradas e repete a **média do preço no treino** para todas as linhas do teste. Os dois modelos são avaliados nas mesmas 3.096 linhas.

$$RMSE_{US\$}=\sqrt{\mathrm{média}[(100000\,y-100000\,\hat{y})^2]}.$$

RMSE pune mais os erros grandes; não é o erro absoluto médio nem uma margem garantida para cada casa. MAE e R² aparecem apenas como métricas complementares.

In [ ]:
# Avaliação final: nenhuma decisão de treino é tomada a partir daqui.
pred_test_usd = prever_dolares(model, x_test)
y_test_usd = y_test_raw.ravel() * 100_000
media_treino_usd = float(y_train_raw.mean() * 100_000)
baseline_test_usd = np.full(y_test_usd.shape, media_treino_usd)
rmse_test = float(np.sqrt(mean_squared_error(y_test_usd, pred_test_usd)))
rmse_baseline = float(np.sqrt(mean_squared_error(y_test_usd, baseline_test_usd)))
reducao_percentual = 100 * (1 - rmse_test / rmse_baseline)
resultados = pd.DataFrame([
    {'Modelo': 'Regressão linear PyTorch', 'RMSE (US$)': rmse_test,
     'MAE (US$)': mean_absolute_error(y_test_usd, pred_test_usd), 'R²': r2_score(y_test_usd, pred_test_usd)},
    {'Modelo': 'Baseline: média do treino', 'RMSE (US$)': rmse_baseline,
     'MAE (US$)': mean_absolute_error(y_test_usd, baseline_test_usd), 'R²': r2_score(y_test_usd, baseline_test_usd)}
])
display(resultados.round(4))
print(f'Média do preço no treino: US$ {media_treino_usd:,.2f}')
print(f'RMSE no teste — modelo: US$ {rmse_test:,.2f} | baseline: US$ {rmse_baseline:,.2f}')
print(f'Redução do RMSE frente ao baseline: {reducao_percentual:.2f}%')
print(f'RMSE / média do treino (contexto de escala, não MAPE): {100 * rmse_test / media_treino_usd:.2f}%')

**Interpretação:** superar o baseline mostra que as entradas acrescentam capacidade preditiva em relação a repetir a média. Isso não basta para declarar o modelo pronto para avaliar imóveis reais: o erro em dólares e o uso pretendido também importam. Os valores históricos e agregados limitam a aplicação a casas individuais e a preços atuais.

### Exemplo prático — previsão de um registro reservado

Usamos a primeira linha do teste para ilustrar o caminho completo: oito valores originais → scaler de X aprendido no treino → modelo → inversão do scaler de y → multiplicação por 100.000. O valor real é mostrado para comparação, sem alterar o modelo.

In [ ]:
exemplo = X.iloc[[idx_test[0]]]
exemplo_tensor = tensor_padronizado(x_scaler, exemplo.to_numpy())
preco_exemplo = float(prever_dolares(model, exemplo_tensor)[0])
display(exemplo)
display(pd.DataFrame({'Preço observado (US$)': [y_test_usd[0]],
                      'Preço previsto (US$)': [preco_exemplo],
                      'Erro previsto − observado (US$)': [preco_exemplo - y_test_usd[0]]}).round(2))

### Diagnóstico dos erros no teste

O resíduo é `observado − previsto`: positivo significa subestimação; negativo, superestimação. Uma boa nuvem ficaria perto do zero sem padrão evidente. O diagnóstico é descritivo: propostas de melhoria precisam de nova validação, sem escolher modelos por este teste.

In [ ]:
residuos = y_test_usd - pred_test_usd
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(pred_test_usd, residuos, s=9, alpha=0.22, color='#2266aa')
axes[0].axhline(0, color='black', linestyle='--')
axes[0].set(xlabel='Valor previsto (US$)', ylabel='Observado − previsto (US$)', title='Resíduos no teste')
axes[1].bar(['Modelo linear', 'Média do treino'], [rmse_test, rmse_baseline], color=['#2266aa', '#777777'])
axes[1].set(ylabel='RMSE no teste (US$)', title='Comparação: menor é melhor')
for i, v in enumerate([rmse_test, rmse_baseline]):
    axes[1].text(i, v, f'US$ {v:,.0f}', ha='center', va='bottom')
axes[1].set_ylim(0, max(rmse_test, rmse_baseline) * 1.18)
for ax in axes:
    ax.ticklabel_format(style='plain', axis='y')
    ax.grid(axis='y', alpha=0.2)
fig.tight_layout()
plt.show()
print('Previsões negativas, sem corte artificial:', int((pred_test_usd < 0).sum()))

**Leitura dos diagnósticos:** o gráfico de resíduos permite procurar tendências e erros extremos; a barra da média é a referência que o modelo deve superar. Mantivemos as previsões originais, inclusive negativas se ocorrerem: cortar valores após ver o teste alteraria a avaliação. A regressão linear não impõe limites físicos aos preços.

### Conferência do pipeline e conclusão calculada

As verificações abaixo conferem formatos, ausência de sobreposição, origem dos scalers e coerência entre RMSE padronizado e RMSE em dólares. A comparação com mínimos quadrados é somente uma checagem numérica no **treino**, sem substituir o modelo PyTorch nem selecionar pelo teste.

In [ ]:
assert (len(idx_train), len(idx_val), len(idx_test)) == (14448, 3096, 3096)
assert len(pesos) == 8 and torch.isfinite(model.weight).all()
assert len(train_losses) == len(val_losses) == n_epochs
assert np.isclose(validate_epoch(model, criterion, x_val, y_val), min(val_losses), atol=1e-6)
rmse_convertido = np.sqrt(validate_epoch(model, criterion, x_test, y_test)) * y_scaler.scale_[0] * 100_000
np.testing.assert_allclose(rmse_test, rmse_convertido, rtol=1e-5)
design = np.column_stack([np.ones(len(x_train)), x_train.numpy()])
theta_ols = np.linalg.lstsq(design, y_train.numpy(), rcond=None)[0].ravel()
mse_ols_train = np.mean((design @ theta_ols - y_train.numpy().ravel()) ** 2)
mse_torch_train = validate_epoch(model, criterion, x_train, y_train)
print(f'MSE de treino — PyTorch selecionado: {mse_torch_train:.6f}; mínimos quadrados: {mse_ols_train:.6f}')
print('Diferença de MSE no treino:', float(mse_torch_train - mse_ols_train))
print('Conferências do pipeline: OK')
display(Markdown(f'**Conclusão dos resultados.** O modelo obteve RMSE de **US$ {rmse_test:,.2f}** '
    f'no teste, contra **US$ {rmse_baseline:,.2f}** da média do treino, uma redução de '
    f'**{reducao_percentual:.2f}%**. O R² no teste foi **{r2_score(y_test_usd, pred_test_usd):.3f}**. '
    f'A época selecionada pela validação foi **{best_epoch}**. '
    f'O maior peso positivo é de **{positivos.iloc[0]["Variável"]}** e o mais negativo é de '
    f'**{negativos.iloc[0]["Variável"]}**, condicionados às demais entradas.'))

**O modelo é bom o suficiente?** É útil como primeira referência e para o objetivo didático; superar a média não garante precisão suficiente para uma decisão imobiliária. Sem um limite de erro aceitável definido, não é possível aprová-lo para uso operacional. Além do erro residual, há dados de 1990, agregação por região e teto do alvo.

**O que tentaria melhorar?** Primeiro investigaria extremos e correlações no treino e testaria Ridge e transformações com validação cruzada. Depois compararia interações geográficas, árvores/gradient boosting ou uma rede com camadas ocultas. Todas as transformações seriam ajustadas dentro de cada treino; usaria separação geográfica para avaliar regiões novas e manteria um teste final independente. Essas alternativas não foram treinadas nesta entrega.

## Aprendizados e referências

- `nn.Linear(8, 1)` combina oito entradas com oito pesos e um bias.
- Os dois `StandardScaler` aprendem exclusivamente com o treino.
- A validação escolhe a época; o teste avalia o modelo selecionado.
- A saída de `inverse_transform` ainda está em US$ 100.000: multiplicar por 100.000 é indispensável.
- Comparar pesos padronizados ajuda a interpretar associações condicionais, mas não demonstra causas.

Referências: [notebook 2.6 do professor](https://github.com/vcasadei/Redes-Neurais-CESAR-School/blob/2026.01-A/2%20-%20Regress%C3%A3o%20Linear/2.6_regressao_linear_salarios.ipynb), [California Housing](https://scikit-learn.org/stable/datasets/real_world.html#california-housing-dataset), [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html), [PyTorch Linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html), [MSELoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.MSELoss.html), [SGD](https://docs.pytorch.org/docs/stable/generated/torch.optim.SGD.html).

**Reprodução:** executar todas as células em ordem, em uma sessão limpa. As versões realmente usadas estão na primeira saída e no `requirements.txt`. O primeiro download exige internet. Não é necessário Google Drive montado, login no dataset ou arquivo de dados enviado junto.